## Rendering detours

Reranking produces two encoded polylines per result. This notebook shows the widget layer
that turns them into map overlays: `widgets.route.RouteController` (the `ipyleaflet`
adapter around `RouteEngine`) and `widgets.output_map.ResponseMap`.

A `ResponseMap` owns a `RouteController` as its `route` attribute, so the map itself is the
route state holder. It also inherits from `widgets.input_map.PositionMap`, which renders
[HERE Map Tile API v3](https://www.here.com/docs/bundle/map-tile-api-v3-api-reference/page/index.html)
tiles with the API key of the credentials — the `L`/`S` toggle switches between the
`lite.day` and `satellite.day` styles.

Setting `minimal_detour` before drawing tells the controller to render the bare route
polyline instead of the corridor polygon:

In [ ]:
from here_search_demo.api import API, HTTPSession
from here_search_demo.auth import Credentials
from here_search_demo.widgets.output_map import ResponseMap

credentials = Credentials()
api = API(credentials=credentials)
session = HTTPSession()

response_map = ResponseMap(credentials=credentials, center=(47.16, 6.28), zoom=6, layout={"height": "500px"})
display(response_map)

In [ ]:
import asyncio

route = response_map.route  # a RouteController instance

route.minimal_detour = True  # draw the route polyline, not the corridor polygon
route.set_route_start((48.57436, 7.71556))  # Strasbourg
route.set_route_stop((45.75049, 4.84418))  # Lyon
route.set_route_width(500)

await route.acquisition_task
await asyncio.sleep(1)  # let the pending map rendering tasks settle

# Rendering the route parks the car marker at the route start, so the driver is
# placed afterwards; `search_at_position` follows it.
route.set_current_position((48.03529, 7.37121))  # near Colmar, already on the way

route.has_route, route.waypoints_count, route.search_at_position

`RouteController` delegates every state attribute to its `RouteEngine`, so the search
context is read from the controller exactly as it was read from the engine in the
[route notebook](obm_6_route.ipynb):

In [ ]:
latitude, longitude = route.search_at_position
resp = await api.browse(
    session=session,
    latitude=latitude,
    longitude=longitude,
    categories=["100-1000"],
    polyline=route.search_flexpolyline,
    width=route.width,
    limit=20,
    lang="en",
)
resp.titles

### Reranking against the map's own cache

The ranker is given `route._route_cache`: the route already fetched by the controller is
reused as the `at → stop` baseline, and the per-item via-routes it computes stay available
to the map for the overlays.

In [ ]:
from here_search_demo.detour import DetourRanker

ranker = DetourRanker(
    credentials=credentials,
    at_pos=route.search_at_position,
    stop_pos=route.stop_position,
    route_cache=route._route_cache,
)
reranked = await ranker.rerank(resp)
[item["_detour"]["label"] for item in reranked.data["items"]]

### Displaying the reranked results

`ResponseMap.display()` draws the items as a GeoJSON layer. `SearchState.hydrate()` indexes
them by rank; `SearchState.title_for()` then returns the `_detour` label instead of the
plain title, which is how the result buttons of `OneBoxMap` get annotated:

In [ ]:
response_map.state.hydrate(reranked)
response_map.display(reranked, fit=True)

[response_map.state.title_for(rank) for rank in response_map.state.ranks()]

### Drawing the detour of a selected result

`ResponseMap.click_result()` simulates a marker click. When the map has a route and the
travel-time mode is on, it looks the `("via", at, item, stop)` entry up in the route cache
and calls `RouteController.draw_detour_routes()`. That method decodes the two polylines
and adds a **green** `at → result` leg and a **purple** `result → stop` leg to the map.

No Routing API call is made here: everything was already fetched by the reranking.

In [ ]:
response_map.click_result(0, emit_action=False, recenter=False, show_details=True)
response_map.route._to_detour_pl is not None, response_map.route._from_detour_pl is not None

The same overlays can be drawn directly from the annotations, without going through the
map state — useful when plugging the feature into another rendering stack:

In [ ]:
from flexpolyline import decode

detour = reranked.data["items"][-1]["_detour"]
response_map.route.draw_detour_routes(detour["polyline_from"], detour["polyline_to"])

len(decode(detour["polyline_to"])), len(decode(detour["polyline_from"]))

In [ ]:
response_map.route.clear_detour_routes()

### In the complete application

`widgets.app.OneBoxMap` wires all of the above to the UI:

* the **travel time** checkbox comes from `RouteController.get_route_checkbox_options()`
* `OneBoxMap._get_context()` feeds `polyline`, `width` and `search_at_position` into every
  search request
* `OneBoxMap._handle_result_list_with_detour()` runs the reranking in a task and falls back
  to the original response on error
* toggling *travel time* off restores the un-reranked response without any new request

The [app automation notebook](../obm_5_automate.ipynb) drives that full flow programmatically.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>